# Đọc file đã được extract image feature *.b sắp sếp lại cho đúng thứ tự với df

Cần file df_meta.preprocessed.parquet của step3 phải được tạo ra trước đó

## Bước này tạo files:

- image_feat.npy (đọc file image_features.*.b tìm có text mà không có hình fill giá trị mặc định)

In [40]:
import os
import array
import numpy as np
import pandas as pd
from tqdm import tqdm

In [2]:
PATH = "../data/2023"

In [ ]:
df = pd.read_parquet(os.path.join(PATH, "step3_text_feature_extract", "df_meta.preprocessed.parquet"))

In [7]:
df_meta = df[["itemID", "asin", "title"]].copy()
df_meta.shape

(35997, 3)

In [8]:
df_meta.info()

<class 'pandas.DataFrame'>
RangeIndex: 35997 entries, 0 to 35996
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   itemID  35997 non-null  int64
 1   asin    35997 non-null  str  
 2   title   35997 non-null  str  
dtypes: int64(1), str(2)
memory usage: 5.0 MB


In [9]:
df_meta[:5]

,itemID,asin,title
0,0,B086QM7FVT,"Skip Hop Toddler Step Stool, Double Up"
1,1,B017IQZ9OK,"Boon Spring Countertop Drying Rack, Green (B11..."
2,2,B08FZJ3YHH,Toilet Seat Covers Disposable - 20 Pack - Wate...
3,3,B082WJTFRR,hiccapop Inflatable Toddler Travel Bed with Sa...
4,4,B004JU0H6O,"Dream On Me 3” Square Corner Playmat, Greengua..."


In [14]:
df_filtered_image = pd.read_parquet(os.path.join(PATH, "step4_image_feature_extract", "df_filtered_image.parquet"))
df_filtered_image.drop(columns=["title"], inplace=True)
df_filtered_image.shape

(35997, 2)

In [15]:
df_filtered_image

,asin,image_path
0,B01C4319LO,step0_clean_data/filtered_images/B01C4319LO.jpg
1,B0083SXABC,step0_clean_data/filtered_images/B0083SXABC.jpg
2,B07JM4RK9T,step0_clean_data/filtered_images/B07JM4RK9T.jpg
3,B08F1VWF5P,step0_clean_data/filtered_images/B08F1VWF5P.jpg
4,B01DDDXTA8,step0_clean_data/filtered_images/B01DDDXTA8.jpg
...,...,...
35992,B0CHYPBD2Z,step0_clean_data/filtered_images/B0CHYPBD2Z.jpg
35993,B0BR6CWGKL,step0_clean_data/filtered_images/B0BR6CWGKL.jpg
35994,B0965ZFFHW,step0_clean_data/filtered_images/B0965ZFFHW.jpg
35995,B0C614K38T,step0_clean_data/filtered_images/B0C614K38T.jpg


In [18]:
df_meta_image = pd.merge(df_meta, df_filtered_image, on="asin", how="inner")
df_meta_image.shape

(35997, 4)

In [19]:
df_meta_image

,itemID,asin,title,image_path
0,0,B086QM7FVT,"Skip Hop Toddler Step Stool, Double Up",step0_clean_data/filtered_images/B086QM7FVT.jpg
1,1,B017IQZ9OK,"Boon Spring Countertop Drying Rack, Green (B11...",step0_clean_data/filtered_images/B017IQZ9OK.jpg
2,2,B08FZJ3YHH,Toilet Seat Covers Disposable - 20 Pack - Wate...,step0_clean_data/filtered_images/B08FZJ3YHH.jpg
3,3,B082WJTFRR,hiccapop Inflatable Toddler Travel Bed with Sa...,step0_clean_data/filtered_images/B082WJTFRR.jpg
4,4,B004JU0H6O,"Dream On Me 3” Square Corner Playmat, Greengua...",step0_clean_data/filtered_images/B004JU0H6O.jpg
...,...,...,...,...
35992,35992,B0C86PC31D,FLEEROSE - CPC-Certified Hip Seat Baby Carrier...,step0_clean_data/filtered_images/B0C86PC31D.jpg
35993,35993,B0BR9ZPR81,DEEZOMO Baby Crib Mattress Sheets 1 Pack Premi...,step0_clean_data/filtered_images/B0BR9ZPR81.jpg
35994,35994,B0000634T0,Ecru Folding Hamper,step0_clean_data/filtered_images/B0000634T0.jpg
35995,35995,B0BLT6WD1V,"Muslin Cotton Baby Car Seat Cover, Universal F...",step0_clean_data/filtered_images/B0BLT6WD1V.jpg


In [20]:
df_meta_image.to_parquet(os.path.join(PATH, "step4_image_feature_extract", "df_meta_image.parquet"), index=False)

In [43]:
df_has_image = df_meta_image[df_meta_image["image_path"].notnull()].copy()
df_does_not_have_image = df_meta_image[df_meta_image["image_path"].isnull()].copy()
print("Số lượng item có ảnh: ", df_has_image.shape)
print("Số lượng item không có ảnh: ", df_does_not_have_image.shape)

Số lượng item có ảnh:  (35979, 4)
Số lượng item không có ảnh:  (18, 4)


In [41]:
def read_image_features(path, feature_size):
    if not os.path.exists(path):
        return
    with open(path, "rb") as f:
        while True:
            asin_bytes = f.read(10)
            if not asin_bytes:
                break
            try:
                asin = asin_bytes.decode("utf-8").strip()
                a = array.array("f")
                # Đọc đúng số lượng float bạn yêu cầu
                a.fromfile(f, feature_size)
                yield asin, a.tolist()
            except EOFError:
                break
            except Exception as e:
                print(f"Lỗi tại vị trí {f.tell()}: {e}")
                break

In [25]:
# --- CẤU HÌNH ---
FEATURE_SIZE = 4096
FILE_B_PATH = os.path.join(PATH, "step4_image_feature_extract", "image_feature.vgg16.b")

In [35]:
image_features = read_image_features(FILE_B_PATH, feature_size=FEATURE_SIZE)

In [36]:
# Lấy thử 1 phần tử đầu tiên
first_item = next(image_features)

# Kiểm tra hình dạng
print(f"Kiểu dữ liệu của 1 phần tử: {type(first_item)}")
print(f"Mã ASIN (ID sản phẩm): {first_item[0]}")
print(f"Độ dài vector ảnh: {len(first_item[1])}")
print(f"5 giá trị đầu tiên trong vector: {first_item[1][:5]}")

Kiểu dữ liệu của 1 phần tử: <class 'tuple'>
Mã ASIN (ID sản phẩm): B0BHX2DRHJ
Độ dài vector ảnh: 4096
5 giá trị đầu tiên trong vector: [0.0, 0.0, 0.0, 0.0, 0.0]


In [37]:
first_item[1][:10]

[0.0, 0.0, 0.0, 0.0, 0.0, 2.642094850540161, 0.0, 0.0, 0.0, 0.6189543008804321]

In [42]:
def process_and_save_image_features(df, file_b_path, output_dir, feature_size=4096):
    """
    Xử lý trích xuất feature từ file .b, map vào itemID và xử lý dữ liệu thiếu.
    
    Args:
        df (pd.DataFrame): DataFrame chứa cột 'asin' và 'itemID'.
        file_b_path (str): Đường dẫn đến file .b chứa features.
        output_dir (str): Thư mục lưu file .npy và log.
        feature_size (int): Kích thước vector feature (mặc định VGG16 là 4096).
    """
    num_items = len(df)
    output_npy = os.path.join(output_dir, "image_feat.npy")
    err_log = os.path.join(output_dir, "missed_img_itemIDs.csv")

    # 1. Map ASIN -> itemID (Đảm bảo itemID là kiểu int để làm index)
    map_asin_itemID = dict(zip(df["asin"], df.index)) # Hoặc df["itemID"] nếu itemID chạy từ 0 đến N-1
    
    # 2. Khởi tạo ma trận và biến hỗ trợ
    final_matrix = np.zeros((num_items, feature_size), dtype=np.float32)
    filled_indices = set()
    running_sum = np.zeros(feature_size, dtype=np.float64)

    print(f"🚀 Đang xử lý ảnh cho {num_items} sản phẩm...")

    # 3. Đọc và điền dữ liệu
    # Lưu ý: Hàm read_image_features cần được định nghĩa trước hoặc import vào
    try:
        for asin, feat_list in tqdm(read_image_features(file_b_path, feature_size), 
                                     total=num_items, desc="Mapping features"):
            if asin in map_asin_itemID:
                target_idx = int(map_asin_itemID[asin])
                feat_array = np.array(feat_list, dtype=np.float32)

                final_matrix[target_idx] = feat_array
                filled_indices.add(target_idx)
                running_sum += feat_array
    except Exception as e:
        print(f"❌ Lỗi khi đọc file feature: {e}")
        return None

    # 4. Xử lý dữ liệu thiếu (Imputation bằng Average Vector)
    all_indices = set(range(num_items))
    missing_indices = sorted(list(all_indices - filled_indices))

    if len(filled_indices) > 0:
        avg_vector = (running_sum / len(filled_indices)).astype(np.float32)

        if missing_indices:
            print(f"⚠️ Cảnh báo: Thiếu {len(missing_indices)} ảnh (~{len(missing_indices)/num_items:.1%}).")
            print(f"👉 Đang lấp đầy bằng vector trung bình và lưu log tại: {err_log}")
            
            # Điền nhanh bằng vectorization
            final_matrix[missing_indices] = avg_vector
            
            # Lưu log các ID thiếu
            np.savetxt(err_log, missing_indices, delimiter=",", fmt="%d")
        
        # 5. Lưu kết quả
        np.save(output_npy, final_matrix)
        print(f"✅ Thành công! Ma trận: {final_matrix.shape}")
        return final_matrix
    
    else:
        print("❌ LỖI: Không có bất kỳ ASIN nào khớp giữa file .b và DataFrame!")
        return None

In [45]:
result = process_and_save_image_features(df_meta_image, FILE_B_PATH, os.path.join(PATH, "step4_image_feature_extract"), FEATURE_SIZE)
result.shape

🚀 Đang xử lý ảnh cho 35997 sản phẩm...


Mapping features: 100%|█████████▉| 35979/35997 [00:08<00:00, 4293.10it/s]


⚠️ Cảnh báo: Thiếu 18 ảnh (~0.1%).
👉 Đang lấp đầy bằng vector trung bình và lưu log tại: ../data/2023\step4_image_feature_extract\missed_img_itemIDs.csv
✅ Thành công! Ma trận: (35997, 4096)


(35997, 4096)